> **中文译文** —— 由社区翻译，非上游原文；源文件变更后以英文原版为准。步骤间的逻辑导读为译者补充。

# 溯源追踪（Provenance Tracking，W3C PROV-O）

## 概览

在高风险领域——医疗、法律、金融、科研——知识图谱的可信度，取决于它能不能回答**"这个事实从哪来？"**。Semantica 的 `provenance` 模块为流经流水线的每个实体、关系和块提供审计级、对齐 W3C PROV-O 的溯源。

> 译注：这是 Semantica 的立身主张——"溯源不可断"。改图数据的代码必须保持溯源链完整；本篇讲的就是这条链怎么建、怎么查、怎么防伪。

本 cookbook 你将学会：

- 给实体和关系带上**来源细节**（DOI、页码、逐字引文、置信度）
- 走完一条事实的完整**血缘**（文档 → 块 → 实体 → KG）
- 审计**修订历史**与实体背后的**全部来源**
- **失效**一条事实而不删除它（prov:Invalidation）——纠正也要可证明
- 用链式 SHA-256 校验和验证**防篡改**

### 先看主线

**场景贯穿全篇**：科研团队把两篇论文（带 DOI）的发现灌进知识图谱。监管者后来问：*"鱼类生物量增加 463% 的说法，出自哪篇论文、哪张图、哪句原文？这条事实被更正过吗？"*

六步四段——**带证据 → 能追溯 → 可纠错 → 防篡改**：

**带证据（第 1–2 步）—— 入库即记录**

1. 第 1 步 记实体：DOI、图表位置、逐字引文、置信度，随事实一起入库
2. 第 2 步 记关系：事实之间的连接也是一等的溯源对象

**能追溯（第 3–4 步）—— 问得出、答得上**

3. 第 3 步 走血缘：一条事实的全部履历（每个版本、每次活动、每个经手的 agent）
4. 第 4 步 审计：修订历史回答"更正过吗"，全部来源回答"谁在支撑它"

**可纠错（第 5 步）—— 改错不抹账**

5. 第 5 步 失效：论文被撤稿时 `invalidate` 归档旧态、追加失效记录（谁、为什么）——绝不静默删除

**防篡改（第 6 步）—— 账本防伪**

6. 第 6 步 验链：每条记录带 SHA-256 校验和，且与前一条相链——事后篡改无所遁形

In [ ]:
!pip install -q semantica

In [ ]:
import json
from semantica.provenance import (
    ProvenanceManager,
    compute_checksum,
    verify_checksum,
)

# In-memory storage for this demo; pass storage_path="provenance.db"
# (or a config with provenance.storage_path) for a persistent SQLite backend.
prov = ProvenanceManager()
print("ProvenanceManager ready (in-memory storage)")

## 第 1 步：用审计级来源细节跟踪实体

入库的每个事实都自带证据：**来源标识**（这里是 DOI）、来源内的**位置**（某张图表）、**逐字引文**、抽取器的**置信度**。

In [ ]:
# Finding from paper #1
entry_biomass = prov.track_entity(
    entity_id="claim_biomass_increase",
    source="DOI:10.1371/journal.pone.0023601",
    confidence=0.92,
    source_location="Figure 2",
    source_quote="Total fish biomass increased by 463% ...",
)

# Supporting entity from paper #2
entry_reserve = prov.track_entity(
    entity_id="marine_reserve_1",
    source="DOI:10.1126/science.1088121",
    confidence=0.88,
    source_location="Table 1",
    source_quote="... no-take marine reserve at Cabo Pulmo ...",
)

print("Tracked:", entry_biomass.entity_id, "|", entry_reserve.entity_id)

## 第 2 步：跟踪事实之间的关系

事实很少孤立存在。生物量增加的论断*关于*海洋保护区——这条关系本身也是一等的溯源对象。

`track_relationship()` 没有专门的 subject/object 字段，按惯例把连接的两个实体记进 `metadata`。

In [ ]:
rel = prov.track_relationship(
    relationship_id="rel_biomass_about_reserve",
    source="DOI:10.1371/journal.pone.0023601",
    metadata={
        "type": "measured_at",
        # No dedicated endpoint fields on track_relationship() yet -- record
        # which entities this relationship connects here by convention.
        "subject_entity_id": "claim_biomass_increase",
        "object_entity_id": "marine_reserve_1",
    },
)

print("Relationship tracked:", rel.entity_id, "|", rel.metadata["subject_entity_id"], "->", rel.metadata["object_entity_id"])

## 第 3 步：走血缘

`get_lineage` 重建一条事实的全部已知信息；`trace_lineage` 返回有序的 `ProvenanceEntry` 链——每个版本、每次活动、每个经手的 agent。

In [ ]:
lineage = prov.get_lineage("claim_biomass_increase")
print(json.dumps(lineage, indent=2, default=str)[:800])

print("\n--- ordered chain ---")
for e in prov.trace_lineage("claim_biomass_increase"):
    print(f"{e.entity_id} | seq#{e.sequence_id} | {e.activity_id}")

## 第 4 步：审计来源与修订历史

监管者问"这条事实更正过吗"时，`revision_history` 给出完整版本链，`get_all_sources` 列出曾经支撑过它的每一份来源文档。

In [ ]:
revisions = prov.revision_history("claim_biomass_increase")
print(f"{len(revisions)} revision(s) on record")

for s in prov.get_all_sources("claim_biomass_increase"):
    print("source:", s)

## 第 5 步：失效——纠正而不删除

假设论文 #1 被部分撤稿。审计链**绝不**静默删除事实：`invalidate` 会归档失效前的状态，并追加一条新的 `prov:Invalidation` 记录，写明**谁**撤的、**为什么**。

In [ ]:
invalidated = prov.invalidate(
    entity_id="claim_biomass_increase",
    agent_id="reviewer_dr_chen",
    reason="Partial retraction: Figure 2 statistics corrected by publisher (see erratum).",
)
print("Invalidated:", invalidated.entity_id, "| invalidated flag:", getattr(invalidated, "invalidated", True))

stats = prov.get_statistics()
print("\nStorage statistics:", json.dumps(stats, indent=2, default=str))

## 第 6 步：验证防篡改

每条记录带确定性的 SHA-256 校验和，且与前一条链式相扣。重算一遍再比对，任何事后篡改都会现形。

In [ ]:
# entry_biomass was returned by track_entity in Step 1
ok = verify_checksum(entry_biomass)
print("Checksum verified:", ok)

print("Computed:", compute_checksum(entry_biomass)[:16], "...")
print("Stored:  ", entry_biomass.checksum[:16] if getattr(entry_biomass, 'checksum', None) else "(see entry fields)")
chain = prov.verify_chain()
print("Chain verification:", json.dumps(chain, default=str)[:200])


## 小结

### 回顾主线

四段闭环：**带证据**（DOI、位置、引文随事实入库）、**能追溯**（血缘与修订历史，问得出就答得上）、**可纠错**（失效不删除，纠正留痕——与 17 篇"没有静默覆盖"同一纪律）、**防篡改**（链式校验和）。监管者的三连问，每一步都有 API 直接回答。

| 需求 | 调用 |
|---|---|
| 记录事实的证据 | `prov.track_entity(entity_id, source, confidence=..., source_location=..., source_quote=...)` |
| 记录关系 | `prov.track_relationship(relationship_id, source, metadata=...)` |
| 事实的完整血缘 | `prov.get_lineage(entity_id)` / `prov.trace_lineage(entity_id)` |
| "更正过吗？" | `prov.revision_history(entity_id)` |
| "哪些来源支撑它？" | `prov.get_all_sources(entity_id)` |
| 撤回而不删除 | `prov.invalidate(entity_id, agent_id, reason=...)` |
| 防篡改检查 | `verify_checksum(entry)` |

### 下一步去哪

- **冲突检测与解决**（notebook 17）——两个来源说法不一时怎么办
- **你的第一张知识图谱**（notebook 08）——给抽取器传 `provenance=True`，摄取时自动跟踪
- 模块 docstring（`help(semantica.provenance)`）记载了与 `kg`、`split`、`conflicts` 跟踪器的可选集成